To chat with the Gemini API, we first need to import the necessary libraries and retrieve your API key from Colab's secrets manager.

In [1]:
import google.generativeai as genai
from google.colab import userdata

# Retrieve the API key from Colab secrets
GEMINI_API_KEY = userdata.get('GEMINI')

# Configure the API client with your key
genai.configure(api_key=GEMINI_API_KEY)

# List available models to find one that supports chat
print('Available models that support `generateContent`:')
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Available models that support `generateContent`:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyri

Now that the API is configured, you can create a generative model instance and start a chat session. Let's try sending a simple message.

In [2]:
# Initialize the Generative Model
model = genai.GenerativeModel('gemini-3.5-flash')

# Start a chat session
chat = model.start_chat(history=[])

# Send a message and print the response
response = chat.send_message("Hello, Gemini! How are you today?")
print(response.text)

Hello! I'm doing great, thank you for asking. I'm ready and excited to help you today. 

How are you doing? What is on your mind, or how can I help you today?


In [3]:
!unzip audiorecord.zip

Archive:  audiorecord.zip
   creating: audiorecord/
  inflating: audiorecord/DMF.m4a     
  inflating: audiorecord/EOT.m4a     
  inflating: audiorecord/kb1.m4a     
  inflating: audiorecord/LICS.m4a    
  inflating: audiorecord/nbdid.m4a   
  inflating: audiorecord/OOVB.m4a    
  inflating: audiorecord/PC.m4a      
  inflating: audiorecord/PE.m4a      


In [4]:
!pip install faster-whisper google-genai pandas librosa torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 80.8 MB/s eta 0:00:00


Whisper STT with Gemini as a post-processing & context-aware layer

In [8]:
import os
import time
import pandas as pd
from google import genai
from google.genai import types
from faster_whisper import WhisperModel
import torch # Import torch to check for CUDA availability

# ------------------------------------------------------------------
# 1. Initialize Gemini Client & Whisper Model
# ------------------------------------------------------------------

MODEL_SIZE = "large-v3"

# Check for CUDA availability and set device accordingly
if torch.cuda.is_available():
    DEVICE = "cuda"
    COMPUTE_TYPE = "float16" # float16 is faster on GPUs
    print("CUDA is available. Using GPU.")
elif torch.backends.mps.is_available():
    DEVICE = "mps" # For Apple Silicon Macs
    COMPUTE_TYPE = "float16"
    print("MPS is available. Using Apple Silicon GPU.")
else:
    DEVICE = "cpu"
    COMPUTE_TYPE = "int8" # int8 is more memory efficient on CPUs
    print("CUDA/MPS not available. Falling back to CPU.")

# The API key is already configured by `genai.configure` in an earlier cell.
# No need to set GEMINI_API_KEY or os.environ["GEMINI_API_KEY"] here.

print("Initializing Gemini Client...")
client = genai.Client(api_key=GEMINI_API_KEY)

print(f"Loading Whisper model '{MODEL_SIZE}' on {DEVICE} with {COMPUTE_TYPE} compute type...")
whisper_model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)

# ------------------------------------------------------------------
# 2. Context-Aware LLM Corrector Function
# ------------------------------------------------------------------
def correct_transcript_with_gemini(raw_transcript: str, bot_context: str) -> str:
    """
    Uses Gemini to post-correct Whisper transcripts using conversational context.
    """
    system_instruction = """
You are an expert Speech-to-Text (STT) post-corrector for Hindi and Hinglish voicebots.
Your job is to fix phonetic transcription errors, misheard numbers, and misaligned domain terms based on the bot's last question.

Rules:
1. Preserve the speaker's original language, intent, and conversational style.
2. If the bot asked for specific slots (e.g., amount, phone number, date, policy number, order ID), correct acoustically confused words into standard digits or target terms.
3. Return ONLY the corrected final transcript without conversational filler or explanations.
"""

    prompt = f"""
Bot's Previous Question/Prompt: "{bot_context}"
Raw Speech Transcript (Whisper Output): "{raw_transcript}"

Corrected Speech Transcript:
"""

    try:
        response = client.models.generate_content(
            model='gemini-3.1-flash-lite', # Updated model to gemini-3.5-flash as requested
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.1  # Low temperature for precise correction
            )
        )
        return response.text.strip()
    except Exception as e:
        print(f"Gemini correction failed: {e}")
        return raw_transcript

# ------------------------------------------------------------------
# 3. Audio Processing Pipeline (No-Context vs. Context-Aware)
# ------------------------------------------------------------------
def process_with_bot_context(audio_path: str, bot_context: str):
    start_time = time.time()

    # Step A: Standard Whisper Baseline (No Context)
    base_segments, _ = whisper_model.transcribe(audio_path, beam_size=5)
    raw_baseline_text = " ".join([s.text.strip() for s in base_segments])

    # Step B: Context-Biased Whisper Transcription
    # Inject bot context into initial_prompt to bias Whisper's acoustic decoder
    whisper_prompt = f"The user is responding to the following query: {bot_context}"

    context_segments, _ = whisper_model.transcribe(
        audio_path,
        beam_size=5,
        initial_prompt=whisper_prompt
    )
    raw_context_text = " ".join([s.text.strip() for s in context_segments])

    # Step C: Gemini Post-Correction Layer
    gemini_corrected_text = correct_transcript_with_gemini(raw_context_text, bot_context)

    elapsed = time.time() - start_time

    return {
        "File": os.path.basename(audio_path),
        "Bot Context / Question": bot_context,
        "1. Raw Baseline (No Context)": raw_baseline_text,
        "2. Whisper + Context Prompt": raw_context_text,
        "3. Whisper + Gemini LLM Post-Corrected": gemini_corrected_text,
        "Inference Time (s)": round(elapsed, 2)
    }

# ------------------------------------------------------------------
# 4. Simulation Test Cases (Call Center Dialog Scenarios)
# ------------------------------------------------------------------
# Map your audio files to expected bot dialog contexts
TEST_SCENARIOS = {
    "DMF.m4a": "Please tell me the date and the amount you transferred.",
    "nbdid.m4a": "Can you provide your 6-digit Order ID?",
    "EOT.m4a": "Please confirm your account number.",
    "OOVB.m4a": "What issue are you facing with your cashback or refund?",
    "LICS.m4a": "How can I help you with your transaction status today?"
}

VALID_EXT = ('.mp3', '.wav', '.m4a', '.flac')

AUDIO_DIR = "/content/audiorecord" # Corrected path

audio_files = [f for f in os.listdir(AUDIO_DIR) if f.lower().endswith(VALID_EXT)] if os.path.exists(AUDIO_DIR) else []

results = []
for file_name in audio_files:
    audio_path = os.path.join(AUDIO_DIR, file_name)

    # Retrieve simulated bot question or use default
    context_question = TEST_SCENARIOS.get(file_name, "What is your query?")

    res = process_with_bot_context(audio_path, context_question)
    results.append(res)

df = pd.DataFrame(results)

print("\n=== Whisper + Gemini Context-Aware Transcription Results ===")
display(df[["File", "Bot Context / Question", "1. Raw Baseline (No Context)", "3. Whisper + Gemini LLM Post-Corrected"]])

df.to_csv("/content/whisper_gemini_context_results.csv", index=False)

CUDA is available. Using GPU.
Initializing Gemini Client...
Loading Whisper model 'large-v3' on cuda with float16 compute type...

=== Whisper + Gemini Context-Aware Transcription Results ===


,File,Bot Context / Question,1. Raw Baseline (No Context),3. Whisper + Gemini LLM Post-Corrected
0,PC.m4a,What is your query?,मेरा काटा बुक पर बाही कटा सिंक नहीं हो रहा,मेरा खाता बुक पर बही खाता सिंक नहीं हो रहा।
1,kb1.m4a,What is your query?,मेरी स्रावनित्रों प्लान एको नहीं हुआ है। रुपिय...,मेरी सबस्क्रिप्शन प्लान एक्टिव नहीं हुआ है। रु...
2,LICS.m4a,How can I help you with your transaction statu...,"मेरा refund process start हुआ है नहीं, status ...","मेरा रिफंड प्रोसेस स्टार्ट हुआ है या नहीं, स्ट..."
3,PE.m4a,What is your query?,आहा आप लोग मेरा कॉल दो बार कट कर चुका हो मैनेज...,आप लोग मेरा कॉल दो बार कट कर चुके हो। मैनेजर स...
4,nbdid.m4a,Can you provide your 6-digit Order ID?,"हाँ, order id है 984450","हाँ, Order ID है 984450"
5,EOT.m4a,Please confirm your account number.,मेरा एकॉंट नमबर है 492810,मेरा अकाउंट नंबर है 492810
6,DMF.m4a,Please tell me the date and the amount you tra...,मैने 25 सेटेंबर 2026 को 3500 रूपीज सेंट किया टी,मैंने 25 सितंबर 2026 को 3500 रुपये सेंड किए।
7,OOVB.m4a,What issue are you facing with your cashback o...,मेरा सर्चेक्स कॉर्ड का कैसबेक क्लेव रिसेक्ट हो...,मेरा सर्विस कार्ड का कैशबैक क्लेम रिजेक्ट हो ग...


below cell is combines hotwords boosting (hotwords), contextual initial prompting (initial_prompt), and Gemini post-processing into a single pipeline.

In [11]:
import os
import time
import pandas as pd
import torch
from google import genai
from google.genai import types
from faster_whisper import WhisperModel

# ------------------------------------------------------------------
# 1. Initialize Gemini Client & Whisper Model
# ------------------------------------------------------------------

MODEL_SIZE = "large-v3"

# Check for hardware acceleration
if torch.cuda.is_available():
    DEVICE = "cuda"
    COMPUTE_TYPE = "float16"
    print("CUDA is available. Using GPU.")
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    COMPUTE_TYPE = "float16"
    print("MPS is available. Using Apple Silicon GPU.")
else:
    DEVICE = "cpu"
    COMPUTE_TYPE = "int8"
    print("CUDA/MPS not available. Falling back to CPU.")

print("Initializing Gemini Client...")
client = genai.Client(api_key=GEMINI_API_KEY)

print(f"Loading Whisper model '{MODEL_SIZE}' on {DEVICE} with {COMPUTE_TYPE} compute type...")
whisper_model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)

# ------------------------------------------------------------------
# 2. Keyword Boosting Setup
# ------------------------------------------------------------------
BOOST_KEYWORDS = [
    "SraVaani Pro",
    "UPI",
    "SurgeX",
    "Khatabook",
    "bahi-khata",
    "25th September 2026",
    "3500 rupees",
    "cashback claim",
    "refund process",
    "manager",
    "492810",
    "984450"
]

# Format keywords into a space-separated hotwords string for faster-whisper
HOTWORDS_STR = " ".join(BOOST_KEYWORDS)

# ------------------------------------------------------------------
# 3. Context-Aware LLM Corrector Function
# ------------------------------------------------------------------
def correct_transcript_with_gemini(raw_transcript: str, bot_context: str, keywords: list) -> str:
    """
    Uses Gemini to post-correct Whisper transcripts using conversational context and target keywords.
    """
    system_instruction = f"""
You are an expert Speech-to-Text (STT) post-corrector for Hindi and Hinglish voicebots.
Your job is to fix phonetic transcription errors, misheard numbers, and misaligned domain terms based on the bot's last question and expected target terms.

Expected Target Vocabulary:
{', '.join(keywords)}

Rules:
1. Preserve the speaker's original language, intent, and conversational style.
2. If the bot asked for specific slots (e.g., amount, phone number, date, policy number, order ID), correct acoustically confused words into standard digits or target terms.
3. If an acoustically similar word matches one of the expected target vocabulary items, map it correctly.
4. Return ONLY the corrected final transcript without conversational filler or explanations.
"""

    prompt = f"""
Bot's Previous Question/Prompt: "{bot_context}"
Raw Speech Transcript (Whisper Output): "{raw_transcript}"

Corrected Speech Transcript:
"""

    try:
        response = client.models.generate_content(
            model='gemini-3.1-flash-lite',
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.1
            )
        )
        return response.text.strip()
    except Exception as e:
        print(f"Gemini correction failed: {e}")
        return raw_transcript

# ------------------------------------------------------------------
# 4. Audio Processing Pipeline (Baseline vs Hotwords + Prompt + Gemini)
# ------------------------------------------------------------------
def process_with_bot_context_and_hotwords(audio_path: str, bot_context: str):
    start_time = time.time()

    # Step A: Standard Whisper Baseline (No Context, No Hotwords)
    base_segments, _ = whisper_model.transcribe(audio_path, beam_size=5)
    raw_baseline_text = " ".join([s.text.strip() for s in base_segments])

    # Step B: Combined Hotwords + Bot Context Prompt in Whisper
    whisper_prompt = f"The user is responding to the query: '{bot_context}'. Relevant terms: {HOTWORDS_STR}."

    boosted_segments, _ = whisper_model.transcribe(
        audio_path,
        beam_size=5,
        hotwords=HOTWORDS_STR,
        initial_prompt=whisper_prompt
    )
    raw_boosted_text = " ".join([s.text.strip() for s in boosted_segments])

    # Step C: Gemini Post-Correction Layer
    gemini_corrected_text = correct_transcript_with_gemini(
        raw_boosted_text,
        bot_context,
        BOOST_KEYWORDS
    )

    elapsed = time.time() - start_time

    return {
        "File": os.path.basename(audio_path),
        "Bot Context / Question": bot_context,
        "1. Raw Baseline": raw_baseline_text,
        "2. Whisper (Hotwords + Prompt)": raw_boosted_text,
        "3. Final (Whisper + Gemini)": gemini_corrected_text,
        "Inference Time (s)": round(elapsed, 2)
    }

# ------------------------------------------------------------------
# 5. Execution
# ------------------------------------------------------------------
TEST_SCENARIOS = {
    "DMF.m4a": "Please tell me the date and the amount you transferred.",
    "nbdid.m4a": "Can you provide your 6-digit Order ID?",
    "EOT.m4a": "Please confirm your account number.",
    "OOVB.m4a": "What issue are you facing with your cashback or refund?",
    "LICS.m4a": "How can I help you with your transaction status today?",
    "PC.m4a": "Are you having an issue with Khatabook sync?",
    "kb1.m4a": "Which plan activation failed?",
    "PE.m4a": "Do you want to speak with a manager?"
}

AUDIO_DIR = "/content/audiorecord"
VALID_EXT = ('.mp3', '.wav', '.m4a', '.flac')

audio_files = [f for f in os.listdir(AUDIO_DIR) if f.lower().endswith(VALID_EXT)] if os.path.exists(AUDIO_DIR) else []

results = []
for file_name in audio_files:
    audio_path = os.path.join(AUDIO_DIR, file_name)
    context_question = TEST_SCENARIOS.get(file_name, "What is your query?")

    res = process_with_bot_context_and_hotwords(audio_path, context_question)
    results.append(res)

df = pd.DataFrame(results)

print("\n=== Whisper (Hotwords + Prompt) + Gemini Results ===")
display(df[["File", "Bot Context / Question", "1. Raw Baseline", "2. Whisper (Hotwords + Prompt)", "3. Final (Whisper + Gemini)"]])

df.to_csv("/content/whisper_hotwords_gemini_results.csv", index=False)

CUDA is available. Using GPU.
Initializing Gemini Client...
Loading Whisper model 'large-v3' on cuda with float16 compute type...

=== Whisper (Hotwords + Prompt) + Gemini Results ===


,File,Bot Context / Question,1. Raw Baseline,2. Whisper (Hotwords + Prompt),3. Final (Whisper + Gemini)
0,PC.m4a,Are you having an issue with Khatabook sync?,मेरा काटा बुक पर बाही कटा सिंक नहीं हो रहा,मेरा काटा बुक पर बाही कटा सिंक नहीं हो रहा।,मेरा Khatabook पर bahi-khata सिंक नहीं हो रहा।
1,kb1.m4a,Which plan activation failed?,मेरी स्रावनित्रों प्लान एको नहीं हुआ है। रुपिय...,मेरी स्रावनिक प्रूप प्लान एक्व नहीं हुआ है। रु...,मेरी SraVaani Pro प्लान एक्टिवेट नहीं हुआ है। ...
2,LICS.m4a,How can I help you with your transaction statu...,"मेरा refund process start हुआ है नहीं, status ...",मेरा refund process स्टार्ट हुआ है नहीं। statu...,मेरा refund process स्टार्ट हुआ है या नहीं? st...
3,PE.m4a,Do you want to speak with a manager?,आहा आप लोग मेरा कॉल दो बार कट कर चुका हो मैनेज...,Aha! आप लोग मेरा कॉल दो बार कट कर चुका हो। Man...,"हाँ, आप लोगों ने मेरा कॉल दो बार कट कर दिया है..."
4,nbdid.m4a,Can you provide your 6-digit Order ID?,"हाँ, order id है 984450",The user is responding to the query. Relevant ...,984450
5,EOT.m4a,Please confirm your account number.,मेरा एकॉंट नमबर है 492810,My account number is 492810.,492810
6,DMF.m4a,Please tell me the date and the amount you tra...,मैने 25 सेटेंबर 2026 को 3500 रूपीज सेंट किया टी,Mayne 25th September 2026 3500 rupees sent KRT.,Maine 25th September 2026 ko 3500 rupees trans...
7,OOVB.m4a,What issue are you facing with your cashback o...,मेरा सर्चेक्स कॉर्ड का कैसबेक क्लेव रिसेक्ट हो...,मेरा सर्चिक्स कॉर्ड का कैस बैक क्लेव रिसेक्ट ह...,मेरा SurgeX कार्ड का cashback claim रिजेक्ट हो...


This code integrates Whisper transcription, Silero VAD turn metrics, confidence scoring, YAMNet event tagging, and Gemini LLM post-processing for intent extraction and phonetic normalization.

In [12]:
import os
import time
import json
import pandas as pd
import numpy as np
import torch
import librosa
import tensorflow as tf
import tensorflow_hub as hub
from google import genai
from google.genai import types
from faster_whisper import WhisperModel

# ------------------------------------------------------------------
# 1. Initialize Hardware & Models
# ------------------------------------------------------------------
MODEL_SIZE = "large-v3"
AUDIO_DIR = "/content/audiorecord"

if torch.cuda.is_available():
    DEVICE = "cuda"
    COMPUTE_TYPE = "float16"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    COMPUTE_TYPE = "float16"
else:
    DEVICE = "cpu"
    COMPUTE_TYPE = "int8"

print("Initializing Gemini Client...")
client = genai.Client(api_key=GEMINI_API_KEY)

print(f"Loading Whisper model '{MODEL_SIZE}' on {DEVICE}...")
whisper_model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)

print("Loading Silero VAD Model...")
vad_model, vad_utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    force_reload=False,
    onnx=False
)
(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = vad_utils

print("Loading YAMNet Audio Event Tagger...")
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')
class_map_path = yamnet_model.class_map_path().numpy().decode('utf-8')
class_names = [line.split(',')[2].strip('"') for line in open(class_map_path).readlines()[1:]]

# ------------------------------------------------------------------
# 2. LLM Post-Processing (Entity Resolution & Intent Parsing)
# ------------------------------------------------------------------
def analyze_with_gemini(raw_transcript: str, bot_context: str, is_stt_unsure: bool) -> dict:
    """
    Performs slot extraction, phonetic normalization, disfluency cleanup,
    and confidence flagging using Gemini 2.5 Flash.
    """
    system_instruction = """
You are an intelligent NLP module for an Indian Hinglish voicebot.
Your job is to analyze raw speech transcripts, fix phonetic errors, remove filler words,
and extract intent and slots into structured JSON format.

JSON Response Schema:
{
  "cleaned_transcript": "<Fixed text without fillers>",
  "intent": "<payment_status | refund_issue | account_inquiry | general_query>",
  "entities": {
    "amount": "<extracted amount or null>",
    "order_id": "<extracted order/account ID or null>",
    "brand_or_product": "<Khatabook | SurgeX | SraVaani Pro | UPI | null>"
  },
  "requires_confirmation": <true if STT was unsure or slots are ambiguous, else false>
}
"""

    prompt = f"""
Bot Context / Prompt Asked: "{bot_context}"
Raw Speech Transcript: "{raw_transcript}"
STT Low Confidence Flag: {is_stt_unsure}

Return the structured JSON output:
"""

    try:
        response = client.models.generate_content(
            model='gemini-3.1-flash-lite',
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.1,
                response_mime_type="application/json"
            )
        )
        return json.loads(response.text.strip())
    except Exception as e:
        print(f"Gemini analysis failed: {e}")
        return {
            "cleaned_transcript": raw_transcript,
            "intent": "unknown",
            "entities": {},
            "requires_confirmation": is_stt_unsure
        }

# ------------------------------------------------------------------
# 3. Audio Analytics & Turn Taking Helpers
# ------------------------------------------------------------------
def analyze_silero_vad(audio_path, min_silence_ms=800):
    wav = read_audio(audio_path, sampling_rate=16000)
    duration = len(wav) / 16000.0

    speech_ts = get_speech_timestamps(
        wav, vad_model, sampling_rate=16000,
        min_silence_duration_ms=min_silence_ms, threshold=0.5
    )

    if speech_ts:
        last_speech_end = speech_ts[-1]['end'] / 16000.0
        trailing_silence = max(0.0, duration - last_speech_end)
        finished_speaking = trailing_silence >= (min_silence_ms / 1000.0)
    else:
        trailing_silence = duration
        finished_speaking = True

    return round(duration, 2), round(trailing_silence, 2), finished_speaking

def classify_yamnet_events(audio_path):
    try:
        wav_data, _ = librosa.load(audio_path, sr=16000, mono=True)
        waveform = tf.convert_to_tensor(wav_data, dtype=tf.float32)
        scores, _, _ = yamnet_model(waveform)
        mean_scores = np.mean(scores.numpy(), axis=0)
        top_indices = np.argsort(mean_scores)[::-1][:2]
        return ", ".join([f"{class_names[i]} ({mean_scores[i]:.2f})" for i in top_indices])
    except Exception:
        return "N/A"

# ------------------------------------------------------------------
# 4. End-to-End Execution
# ------------------------------------------------------------------
TEST_SCENARIOS = {
    "DMF.m4a": "Please tell me the date and the amount you transferred.",
    "nbdid.m4a": "Can you provide your 6-digit Order ID?",
    "EOT.m4a": "Please confirm your account number.",
    "OOVB.m4a": "What issue are you facing with your cashback or refund?",
    "LICS.m4a": "How can I help you with your transaction status today?"
}

VALID_EXT = ('.mp3', '.wav', '.m4a', '.flac')
audio_files = [f for f in os.listdir(AUDIO_DIR) if f.lower().endswith(VALID_EXT)] if os.path.exists(AUDIO_DIR) else []

results = []

for file_name in audio_files:
    audio_path = os.path.join(AUDIO_DIR, file_name)
    bot_context = TEST_SCENARIOS.get(file_name, "How can I help you?")

    start_time = time.time()

    # 1. Silero VAD Analysis
    duration, trailing_silence, turn_finished = analyze_silero_vad(audio_path, min_silence_ms=800)

    # 2. Whisper Transcription
    segments, info = whisper_model.transcribe(
        audio_path,
        beam_size=5,
        initial_prompt=f"User is responding to: {bot_context}"
    )
    segment_list = list(segments)
    raw_transcript = " ".join([s.text.strip() for s in segment_list])

    # 3. Confidence Metrics
    avg_logprobs = [s.avg_logprob for s in segment_list] if segment_list else [-99.0]
    avg_confidence = round(float(np.mean(avg_logprobs)), 3)
    is_unsure = avg_confidence < -0.8

    # 4. YAMNet Audio Events
    audio_events = classify_yamnet_events(audio_path)

    # 5. Gemini Structured Post-Processing
    gemini_analysis = analyze_with_gemini(raw_transcript, bot_context, is_unsure)

    elapsed = time.time() - start_time

    results.append({
        "File": file_name,
        "Duration (s)": duration,
        "Trailing Silence (s)": trailing_silence,
        "Turn Finished?": "YES" if turn_finished else "NO",
        "STT LogProb": avg_confidence,
        "STT Unsure Flag": "YES" if is_unsure else "NO",
        "Audio Events": audio_events,
        "Raw Transcript": raw_transcript,
        "Cleaned Transcript": gemini_analysis.get("cleaned_transcript", ""),
        "Extracted Intent": gemini_analysis.get("intent", ""),
        "Extracted Entities": json.dumps(gemini_analysis.get("entities", {})),
        "Confirm Slot?": "YES" if gemini_analysis.get("requires_confirmation") else "NO",
        "Inference Time (s)": round(elapsed, 2)
    })

# Display Results Summary
df = pd.DataFrame(results)
print("\n=== Comprehensive ASR + LLM Voicebot Pipeline Summary ===")
display(df[[
    "File",
    "Turn Finished?",
    "STT Unsure Flag",
    "Raw Transcript",
    "Cleaned Transcript",
    "Extracted Intent",
    "Extracted Entities",
    "Confirm Slot?"
]])

df.to_csv("/content/comprehensive_asr_llm_pipeline_results.csv", index=False)

/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Initializing Gemini Client...
Loading Whisper model 'large-v3' on cuda...
Loading Silero VAD Model...
The repository snakers4_silero-vad does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip
Loading YAMNet Audio Event Tagger...


/tmp/ipykernel_1970/760596259.py:128: UserWarning: PySoundFile failed. Trying audioread instead.
  wav_data, _ = librosa.load(audio_path, sr=16000, mono=True)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_1970/760596259.py:128: UserWarning: PySoundFile failed. Trying audioread instead.
  wav_data, _ = librosa.load(audio_path, sr=16000, mono=True)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_1970/760596259.py:128: UserWarning: PySoundFile failed. Trying audioread instead.
  wav_data, _ = librosa.load(audio_path, sr=160


=== Comprehensive ASR + LLM Voicebot Pipeline Summary ===


,File,Turn Finished?,STT Unsure Flag,Raw Transcript,Cleaned Transcript,Extracted Intent,Extracted Entities,Confirm Slot?
0,PC.m4a,YES,NO,मेरा काटा बुक पर बाही कटा सिंक नहीं हो रहा।,मेरा Khatabook पर बही खाता सिंक नहीं हो रहा है।,account_inquiry,"{""amount"": null, ""order_id"": null, ""brand_or_p...",NO
1,kb1.m4a,YES,NO,मेरी स्रावनित्रों प्लान एको नहीं हुआ है। रुपये...,मेरी SraVaani Pro प्लान एक्टिव नहीं हुआ है। UP...,payment_status,"{""amount"": null, ""order_id"": null, ""brand_or_p...",NO
2,LICS.m4a,YES,NO,मेरा refund process स्टार्ट हुआ है नहीं status...,"मेरा refund process स्टार्ट हुआ है या नहीं, st...",refund_issue,"{""amount"": null, ""order_id"": null, ""brand_or_p...",NO
3,PE.m4a,YES,NO,"आहा, आप लोग मेरा कॉल दो बार कट कर चुका हो। मैन...",आप लोग मेरा कॉल दो बार कट कर चुके हो। मैनेजर स...,general_query,"{""amount"": null, ""order_id"": null, ""brand_or_p...",NO
4,nbdid.m4a,YES,NO,"हाँ, Order ID है 984450",Order ID 984450 hai,account_inquiry,"{""amount"": null, ""order_id"": ""984450"", ""brand_...",NO
5,EOT.m4a,YES,NO,मेरा एकॉंट नंबर है 492810,मेरा अकाउंट नंबर है 492810,account_inquiry,"{""amount"": null, ""order_id"": ""492810"", ""brand_...",NO
6,DMF.m4a,YES,NO,मैंने 25 सेटेंबर 2026 को 3500 रूपीस सेंड किया टी,मैंने 25 सितंबर 2026 को 3500 रुपये सेंड किए,payment_status,"{""amount"": ""3500"", ""order_id"": null, ""brand_or...",NO
7,OOVB.m4a,YES,NO,मेरा सर्चिक्स कॉर्ड का कैस बैक क्लेव रिसेक्ट ह...,मेरा SurgeX कार्ड का कैशबैक क्लेम रिजेक्ट हो ग...,refund_issue,"{""amount"": null, ""order_id"": null, ""brand_or_p...",NO
